# Summary Agent Test - Report 1 (LS2502)

This notebook tests the AR Summary Checker Agent using data from report1.

## Overview
1. Load and extract data from LS2502 - Final Draft R2.docx
2. Parse AR summaries and recommendation data
3. Validate summaries against numerical data
4. Generate AI-powered analysis (if API key available)

## Setup


In [1]:
import os
import json
import sys
from pathlib import Path
from dotenv import load_dotenv

# Add parent directory to path
sys.path.insert(0, '..')

# Import document extraction utilities
from document_extractor import extract_itac_report
from doc_extractor_utils import (
    parse_ar_summaries,
    get_recommended_summary_table_json,
    get_single_ar_summary_table,
    compare_ar_with_summary
)

# Import agents (new ADK-based structure)
from agents.summary_checker import (
    check_all_ar_summaries,
    analyze_with_llm,
    validate_ar_summary,
    get_agent_config,
    create_agent
)

# Load environment variables
load_dotenv()

print("✓ Imports successful!")
print(f"Python version: {sys.version}")


✓ Imports successful!
Python version: 3.10.19 (main, Oct 21 2025, 16:37:10) [Clang 20.1.8 ]


In [2]:
# Display agent configuration
try:
    config = get_agent_config()
    print("📋 Agent Configuration:")
    print(f"  Agent Name: {config['agent']['name']}")
    print(f"  Model: {config['model']['name']}")
    print(f"  Temperature: {config['model']['temperature']}")
    print(f"  Max Output Tokens: {config['model']['max_output_tokens']}")
except Exception as e:
    print(f"⚠️  Could not load config: {e}")


📋 Agent Configuration:
  Agent Name: ar_summary_validator
  Model: gemini-3-flash-preview
  Temperature: 0.7
⚠️  Could not load config: 'max_output_tokens'


## 1. Load Report1 Document

Load the LS2502 document from the docs/report1/ folder and extract HTML data.


In [7]:
# Define path to report1
report1_path = Path("../docs/report1/LS2502 - Final Draft R2.docx")

# Verify file exists
if not report1_path.exists():
    raise FileNotFoundError(f"Report not found: {report1_path}")

print(f"📄 Loading report: {report1_path.name}")
print(f"   File size: {report1_path.stat().st_size / 1024 / 1024:.2f} MB")
print()

# Extract report data
print("🔄 Extracting report data...")
extracted_data = extract_itac_report(
    str(report1_path),
    output="html",
    save_files=True
)

print(f"✓ Extraction complete!")
print(f"  Extracted sections: {list(extracted_data.keys())}")


📄 Loading report: LS2502 - Final Draft R2.docx
   File size: 4.94 MB

🔄 Extracting report data...


INFO:root:Attempting to extract links from 5 AR(s)
INFO:link_extractor:Found 44 footnote elements in document
INFO:link_extractor:Extracted 41 footnotes with content
INFO:link_extractor:Found 7 link(s) in AR_01 (including footnotes)
INFO:link_extractor:Found 3 link(s) in AR_02 (including footnotes)
INFO:link_extractor:Found 6 link(s) in AR_03 (including footnotes)
INFO:link_extractor:Found 14 link(s) in AR_04 (including footnotes)
INFO:link_extractor:Found 6 link(s) in AR_05 (including footnotes)
INFO:root:Link extraction complete: {'total_links': 36, 'total_ars_with_links': 5, 'hyperlink_count': 30, 'text_url_count': 6, 'links_by_ar': {'AR_01': 7, 'AR_02': 3, 'AR_03': 6, 'AR_04': 14, 'AR_05': 6}}


✓ Extraction complete!
  Extracted sections: ['general_information', 'annual_energy_usages_and_costs', 'carbon_footprint', 'recommendation_summary_table', 'ar_summary', 'assessment_recommendations', 'ar_links']


DEBUG: document_extractor.py:546 in build_outputs()
       f"Extracted links from ARs: {link_stats}": 'Extracted links from ARs: {'total_links': 36, 'total_ars_with_links': 5, 'hyperlink_count': 30, 'text_url_count': 6, 'links_by_ar': {'AR_01': 7, 'AR_02': 3, 'AR_03': 6, 'AR_04': 14, 'AR_05': 6}}'


In [8]:
# Display extraction statistics
print("📊 Extraction Statistics:")
print()
for section_name, content in extracted_data.items():
    if isinstance(content, str):
        print(f"  {section_name}: {len(content):,} characters")
    elif isinstance(content, dict):
        print(f"  {section_name}: {len(str(content)):,} characters (dict)")
    elif isinstance(content, list):
        print(f"  {section_name}: {len(content)} items")


📊 Extraction Statistics:

  general_information: 740 characters
  annual_energy_usages_and_costs: 1,819 characters
  carbon_footprint: 1,188 characters
  recommendation_summary_table: 4,414 characters
  ar_summary: 2,019 characters
  assessment_recommendations: 5 items
  ar_links: 17,418 characters (dict)


In [38]:
extracted_data['assessment_recommendations']

['<p>AR No. 1 – HVAC Tune-Up to Increase Energy Efficiency</p>\n<p><i>(ARC Code </i><i>2.7211</i><i>)</i></p>\n<p style="text-align:center">Table 4-1. The Savings Summary for AR No. 1</p>\n<table border=\'1\' cellpadding=\'4\' cellspacing=\'0\' style=\'border-collapse:collapse;width:100%\'><tr><td><p style="text-align:center"><b>Energy Savings (kWh/yr)</b></p></td><td><p style="text-align:center"><b>Energy Cost Savings ($/yr)</b></p></td><td><p style="text-align:center"><b>Total Cost Savings ($/yr)</b></p></td><td><p style="text-align:center"><b>CO</b><b>2</b><b> Reduction (Tons/yr)</b>\xa0</p></td><td><p style="text-align:center"><b>Imp. Cost ($)</b></p></td><td><p style="text-align:center"><b>Payback Period</b></p><p style="text-align:center"><b>(yr)</b></p></td></tr><tr><td><p style="text-align:center">72,020</p></td><td><p style="text-align:center">8,066</p></td><td><p style="text-align:center">8,066</p></td><td><p style="text-align:center">27</p></td><td><p style="text-align:cente

## 2. Parse AR Summaries

Extract and parse the AR summaries from the report.


In [9]:
# Get AR summary HTML
ar_summary_html = extracted_data.get('ar_summary', '')

if not ar_summary_html:
    raise ValueError("No AR summary data found in extracted report")

print(f"📝 AR Summary HTML: {len(ar_summary_html):,} characters")
print()

# Parse AR summaries
ar_summaries = parse_ar_summaries(ar_summary_html)

print(f"✓ Found {len(ar_summaries)} AR summaries")
print()

# Display AR numbers
ar_numbers = [ar['ar_no'] for ar in ar_summaries]
print(f"AR Numbers: {ar_numbers}")


📝 AR Summary HTML: 2,019 characters

✓ Found 5 AR summaries

AR Numbers: [1, 2, 3, 4, 5]


In [10]:
# Display first AR summary as example
if ar_summaries:
    first_ar = ar_summaries[0]
    print("="*70)
    print(f"Example: AR {first_ar['ar_no']}")
    print("="*70)
    print(first_ar['ar_summary'])
    print("="*70)


Example: AR 1
AR No. 1 – HVAC Tune-Up to Increase Energy Efficiency  Performing a tune-up of the HVAC system will optimize its performance and increase energy efficiency. This recommendation includes adjusting and calibrating system controls, cleaning components, and ensuring proper operation. The annual savings from this recommendation are estimated to be $8,06 with no implementation cost


## 3. Parse Recommendation Summary Table

Extract numerical data from the recommendation summary table.


In [11]:
# Get recommendation summary table HTML
rec_summary_html = extracted_data.get('recommendation_summary_table', '')

if not rec_summary_html:
    raise ValueError("No recommendation summary table found in extracted report")

print(f"📊 Recommendation Summary HTML: {len(rec_summary_html):,} characters")
print()

# Parse recommendation summary table
rec_summary_data = get_recommended_summary_table_json(rec_summary_html)
recommendations = rec_summary_data.get('recommendations', [])

print(f"✓ Found {len(recommendations)} recommendations in summary table")
print()

# Display headers
headers = rec_summary_data.get('standardized_headers', [])
print(f"Headers ({len(headers)}):")
for i, header in enumerate(headers, 1):
    print(f"  {i}. {header}")


📊 Recommendation Summary HTML: 4,414 characters

✓ Found 5 recommendations in summary table

Headers (11):
  1. ar_number
  2. category
  3. description
  4. electricity_savings_kwh_per_year
  5. energy_cost_savings_per_year
  6. demand_savings_kw_per_year
  7. demand_cost_savings_per_year
  8. total_cost_savings_per_year
  9. co2_reduction_tons_per_year
  10. implementation_cost
  11. payback_period_years


In [12]:
# Display first recommendation details
if recommendations:
    first_rec = recommendations[0]
    print("="*70)
    print(f"Example: AR {first_rec.get('ar_number')} - Numerical Data")
    print("="*70)
    for key, value in first_rec.items():
        if key not in ['category', 'description']:
            print(f"  {key}: {value}")
    print("="*70)


Example: AR 1 - Numerical Data
  ar_number: 1
  electricity_savings_kwh_per_year: 72020
  energy_cost_savings_per_year: 8066
  demand_savings_kw_per_year: 0
  demand_cost_savings_per_year: 0
  total_cost_savings_per_year: 8066
  co2_reduction_tons_per_year: 27
  implementation_cost: 0
  payback_period_years: 0


In [13]:
recommendations[0]

{'ar_number': 1,
 'category': 'Building and Grounds',
 'description': 'HVAC Tune-Up to Increase Energy Efficiency',
 'electricity_savings_kwh_per_year': 72020,
 'energy_cost_savings_per_year': 8066,
 'demand_savings_kw_per_year': 0,
 'demand_cost_savings_per_year': 0,
 'total_cost_savings_per_year': 8066,
 'co2_reduction_tons_per_year': 27,
 'implementation_cost': 0,
 'payback_period_years': 0}

## 4. Load Individual AR Data

Load detailed data for each individual AR from the extracted HTML files.


In [41]:
extracted_data['assessment_recommendations']

['<p>AR No. 1 – HVAC Tune-Up to Increase Energy Efficiency</p>\n<p><i>(ARC Code </i><i>2.7211</i><i>)</i></p>\n<p style="text-align:center">Table 4-1. The Savings Summary for AR No. 1</p>\n<table border=\'1\' cellpadding=\'4\' cellspacing=\'0\' style=\'border-collapse:collapse;width:100%\'><tr><td><p style="text-align:center"><b>Energy Savings (kWh/yr)</b></p></td><td><p style="text-align:center"><b>Energy Cost Savings ($/yr)</b></p></td><td><p style="text-align:center"><b>Total Cost Savings ($/yr)</b></p></td><td><p style="text-align:center"><b>CO</b><b>2</b><b> Reduction (Tons/yr)</b>\xa0</p></td><td><p style="text-align:center"><b>Imp. Cost ($)</b></p></td><td><p style="text-align:center"><b>Payback Period</b></p><p style="text-align:center"><b>(yr)</b></p></td></tr><tr><td><p style="text-align:center">72,020</p></td><td><p style="text-align:center">8,066</p></td><td><p style="text-align:center">8,066</p></td><td><p style="text-align:center">27</p></td><td><p style="text-align:cente

In [42]:
# Load individual AR data from extracted_data
print("📋 Loading individual AR data from extracted_data['assessment_recommendations']")
print()

# Get AR list from extracted_data
ar_htmls = extracted_data.get('assessment_recommendations', [])

print(f"   Found {len(ar_htmls)} AR sections in extracted data")
print()

ar_data_list = []

for idx, ar_html in enumerate(ar_htmls, 1):
    # Parse AR data
    ar_data = get_single_ar_summary_table(ar_html)
    
    if ar_data.get('ar_number'):
        ar_data_list.append(ar_data)
        data_fields = len(ar_data.get('data', {}))
        print(f"  ✓ AR {ar_data['ar_number']:2d}: {data_fields} data fields")

print()
print(f"✓ Total ARs loaded: {len(ar_data_list)}")

📋 Loading individual AR data from extracted_data['assessment_recommendations']

   Found 5 AR sections in extracted data

  ✓ AR  1: 6 data fields
  ✓ AR  2: 6 data fields
  ✓ AR  3: 6 data fields
  ✓ AR  4: 6 data fields
  ✓ AR  5: 8 data fields

✓ Total ARs loaded: 5


In [43]:
# Display first AR's detailed data
if ar_data_list:
    first_ar_data = ar_data_list[0]
    print("="*70)
    print(f"Example: AR {first_ar_data['ar_number']} - Detailed Data")
    print("="*70)
    print(f"AR Number: {first_ar_data['ar_number']}")
    print(f"\nData Fields:")
    for key, value in first_ar_data.get('data', {}).items():
        print(f"  {key}: {value}")
    print("="*70)


Example: AR 1 - Detailed Data
AR Number: 1

Data Fields:
  electricity_savings_kwh_per_year: 72020
  energy_cost_savings_per_year: 8066
  total_cost_savings_per_year: 8066
  co2_reduction_tons_per_year: 27
  implementation_cost: 0
  payback_period_years: 0.0


## 5. Validate AR Summaries

Run the Summary Checker Agent to validate AR summaries against numerical data.


In [18]:
# Check for API key
api_key = os.getenv('GOOGLE_API_KEY') or os.getenv('GEMINI_API_KEY')

if not api_key:
    print("⚠️  WARNING: GOOGLE_API_KEY or GEMINI_API_KEY not set.")
    print("   Some features may be limited.")
    print("   To enable full functionality, set the environment variable:")
    print("   export GOOGLE_API_KEY='your-api-key-here'")
    print()
else:
    print("✓ API key found")
    print()


✓ API key found



In [70]:
from pprint import pprint
pprint(ar_summaries[0])

{'ar_no': 1,
 'ar_summary': 'AR No. 1 – HVAC Tune-Up to Increase Energy Efficiency  '
               'Performing a tune-up of the HVAC system will optimize its '
               'performance and increase energy efficiency. This '
               'recommendation includes adjusting and calibrating system '
               'controls, cleaning components, and ensuring proper operation. '
               'The annual savings from this recommendation are estimated to '
               'be $8,06 with no implementation cost'}


In [72]:
recommendations[3]

{'ar_number': 4,
 'category': 'Combustion Systems',
 'description': 'Eliminate Leaks in Compressed Air Lines',
 'electricity_savings_kwh_per_year': 8083,
 'energy_cost_savings_per_year': 905,
 'demand_savings_kw_per_year': 0,
 'demand_cost_savings_per_year': 0,
 'total_cost_savings_per_year': 905,
 'co2_reduction_tons_per_year': 3,
 'implementation_cost': 45,
 'payback_period_years': 0.05}

In [47]:
ar_data_list[1]['data']

{'electricity_savings_kwh_per_year': 24928,
 'energy_cost_savings_per_year': 2792,
 'total_cost_savings_per_year': 2792,
 'co2_reduction_tons_per_year': 10,
 'imp_costdollar': 100,
 'payback_period_years': 0.04}

In [48]:
len(recommendations) , len(ar_data_list)

(5, 5)

In [49]:
# Run validation
print("🔍 Validating AR summaries...")
print()

validation_results = check_all_ar_summaries(
    ar_summaries,
    recommendations,
    ar_data_list,
    api_key=api_key
)

print(f"✓ Validated {len(validation_results)} AR summaries")
print()
print("="*70)


🔍 Validating AR summaries...

✓ Validated 5 AR summaries



## 6. Display Validation Results

Show detailed validation results for each AR.


In [50]:
# Display results
print("VALIDATION RESULTS")
print("="*70)
print()

for result in validation_results:
    ar_num = result.get('ar_number')
    
    # Handle errors
    if result.get('status') == 'error':
        print(f"❌ AR {ar_num}: {result.get('message')}")
        print()
        continue
    
    # Get validation details
    validation = result.get('validation', {})
    comparison = result.get('comparison', {})
    
    has_diffs = validation.get('has_differences', False)
    total_matches = validation.get('total_matches', 0)
    total_diffs = validation.get('total_differences', 0)
    
    # Display status
    status_icon = "⚠️ " if has_diffs else "✅"
    print(f"{status_icon} AR {ar_num}:")
    print(f"   Matches: {total_matches}, Differences: {total_diffs}")
    
    # Display discrepancies if any
    if has_diffs:
        print(f"   Discrepancies:")
        for diff in comparison.get('differences', []):
            field = diff.get('field')
            ar_val = diff.get('ar_value')
            summary_val = diff.get('summary_value')
            difference = diff.get('difference')
            print(f"     • {field}:")
            print(f"       AR value: {ar_val}")
            print(f"       Summary value: {summary_val}")
            print(f"       Difference: {difference}")
    print()

print("="*70)


VALIDATION RESULTS

✅ AR 1:
   Matches: 6, Differences: 0

✅ AR 2:
   Matches: 5, Differences: 0

✅ AR 3:
   Matches: 6, Differences: 0

✅ AR 4:
   Matches: 5, Differences: 0

✅ AR 5:
   Matches: 5, Differences: 0



In [51]:
# Summary statistics
total_ars = len(validation_results)
ars_with_issues = sum(1 for r in validation_results 
                      if r.get('validation', {}).get('has_differences', False))
ars_clean = total_ars - ars_with_issues
error_count = sum(1 for r in validation_results if r.get('status') == 'error')

print("📊 SUMMARY STATISTICS")
print("="*70)
print(f"  Total ARs validated: {total_ars}")
print(f"  ARs with discrepancies: {ars_with_issues}")
print(f"  ARs without discrepancies: {ars_clean}")
print(f"  Validation errors: {error_count}")
print()
if total_ars > 0:
    accuracy = (ars_clean / total_ars) * 100
    print(f"  Accuracy: {accuracy:.1f}%")
print("="*70)


📊 SUMMARY STATISTICS
  Total ARs validated: 5
  ARs with discrepancies: 0
  ARs without discrepancies: 5
  Validation errors: 0

  Accuracy: 100.0%


## 7. Save Results

Save validation results to a JSON file for later analysis.


In [52]:
# Save results to JSON
output_file = Path("../ar_summary_validation_report1.json")

with open(output_file, 'w') as f:
    json.dump(validation_results, f, indent=2)

print(f"💾 Validation results saved to: {output_file}")
print(f"   File size: {output_file.stat().st_size / 1024:.2f} KB")


💾 Validation results saved to: ../ar_summary_validation_report1.json
   File size: 17.03 KB


## 8. AI-Powered Analysis (Optional)

Use the LLM to generate a comprehensive analysis of validation results.

**Note:** This requires a valid GOOGLE_API_KEY or GEMINI_API_KEY.


In [53]:
# Generate AI analysis if API key is available
if api_key and validation_results:
    print("🤖 Generating AI-powered analysis using Gemini...")
    print()
    
    try:
        analysis = analyze_with_llm(validation_results, api_key=api_key)
        
        print("="*70)
        print("AI ANALYSIS REPORT")
        print("="*70)
        print()
        print(analysis)
        print()
        print("="*70)
        
        # Save analysis to file
        analysis_file = Path("../ar_summary_analysis_report1.txt")
        with open(analysis_file, 'w') as f:
            f.write(analysis)
        
        print()
        print(f"💾 Analysis saved to: {analysis_file}")
        print(f"   File size: {analysis_file.stat().st_size / 1024:.2f} KB")
        
    except Exception as e:
        print(f"❌ Error during AI analysis: {e}")
        import traceback
        traceback.print_exc()
else:
    print("ℹ️  Skipping AI analysis")
    if not api_key:
        print("   Reason: No API key provided")
        print("   To enable, set GOOGLE_API_KEY or GEMINI_API_KEY environment variable")
    elif not validation_results:
        print("   Reason: No validation results available")


INFO:google_genai.models:AFC is enabled with max remote calls: 10.


🤖 Generating AI-powered analysis using Gemini...



INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3-flash-preview:generateContent "HTTP/1.1 200 OK"


AI ANALYSIS REPORT

This report provides a comprehensive analysis of the validation results for Assessment Recommendations (AR) 1 through 5. The analysis identifies data inconsistencies, evaluates the severity of these issues, and provides actionable recommendations for correction.

---

### 1. Analysis of ARs with Data Inconsistencies

**AR 1: HVAC Tune-Up**
*   **Discrepancy:** The summary text lists annual savings as **$8,06**, while the numerical data record shows **$8,066**.
*   **Missing Field:** The implementation cost is listed as $0 in the data matches but is noted as "no implementation cost" in the text (this is consistent, but the typo in the savings figure is a primary concern).

**AR 2: Compressed Air System Pressure**
*   **Discrepancy:** The summary text states an implementation cost of **$10**. However, the numerical data shows a payback period of **0.04 years**. 
    *   *Calculation:* $2,792 (savings) × 0.04 (payback) = **$111.68**. 
    *   A $10 cost with $2,792 sav

In [ ]:
ar_1_summary = ar_summaries[0]['ar_summary']
ar_number = 1


In [ ]:

initial_validation = validate_ar_summary()

In [54]:
# Function to display detailed AR information
def display_ar_details(ar_number):
    """Display detailed information for a specific AR."""
    print("="*70)
    print(f"DETAILED VIEW: AR {ar_number}")
    print("="*70)
    print()
    
    # Find AR summary
    ar_summary = next((ar for ar in ar_summaries if ar['ar_no'] == ar_number), None)
    if ar_summary:
        print("📝 SUMMARY TEXT:")
        print("-"*70)
        print(ar_summary['ar_summary'])
        print()
    
    # Find AR data
    ar_data = next((ar for ar in ar_data_list if ar['ar_number'] == ar_number), None)
    if ar_data:
        print("📊 NUMERICAL DATA:")
        print("-"*70)
        for key, value in ar_data.get('data', {}).items():
            print(f"  {key}: {value}")
        print()
    
    # Find validation result
    result = next((r for r in validation_results if r.get('ar_number') == ar_number), None)
    if result:
        print("✓ VALIDATION RESULT:")
        print("-"*70)
        validation = result.get('validation', {})
        print(f"  Has differences: {validation.get('has_differences', False)}")
        print(f"  Total matches: {validation.get('total_matches', 0)}")
        print(f"  Total differences: {validation.get('total_differences', 0)}")
        
        if validation.get('has_differences'):
            print(f"\n  Discrepancies:")
            for diff in result.get('comparison', {}).get('differences', []):
                print(f"    • {diff.get('field')}:")
                print(f"      AR: {diff.get('ar_value')}")
                print(f"      Summary: {diff.get('summary_value')}")
                print(f"      Diff: {diff.get('difference')}")
    
    print("="*70)

# Example: Display details for AR 1
# Uncomment to run:
# display_ar_details(1)


In [55]:
# Display all ARs with discrepancies in detail
print("🔍 ARs WITH DISCREPANCIES (Detailed View)")
print()

ars_with_discrepancies = [
    r.get('ar_number') for r in validation_results 
    if r.get('validation', {}).get('has_differences', False)
]

if ars_with_discrepancies:
    for ar_num in ars_with_discrepancies:
        display_ar_details(ar_num)
        print()
else:
    print("✅ No ARs with discrepancies found!")
    print("   All AR summaries are consistent with their numerical data.")


🔍 ARs WITH DISCREPANCIES (Detailed View)

✅ No ARs with discrepancies found!
   All AR summaries are consistent with their numerical data.


## 10. Conclusion

Summary of the testing session.


In [65]:
ar_data =get_single_ar_summary_table(extracted_data['assessment_recommendations'][0])
recommendation_data = get_recommended_summary_table_json(extracted_data['recommendation_summary_table'])


In [68]:
ar_data


{'ar_number': 1,
 'headers': ['Energy Savings (kWh/yr)',
  'Energy Cost Savings ($/yr)',
  'Total Cost Savings ($/yr)',
  'CO2 Reduction (Tons/yr)',
  'Imp. Cost ($)',
  'Payback Period(yr)'],
 'standardized_headers': ['electricity_savings_kwh_per_year',
  'energy_cost_savings_per_year',
  'total_cost_savings_per_year',
  'co2_reduction_tons_per_year',
  'implementation_cost',
  'payback_period_years'],
 'data': {'electricity_savings_kwh_per_year': 72020,
  'energy_cost_savings_per_year': 8066,
  'total_cost_savings_per_year': 8066,
  'co2_reduction_tons_per_year': 27,
  'implementation_cost': 0,
  'payback_period_years': 0.0}}

In [69]:
compare_data = compare_ar_with_summary(ar_data, recommendation_data)
compare_data 

AttributeError: 'str' object has no attribute 'get'